In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
#4. Importar las librerías adecuadas y cargar los datos del archivo en una variable llamada 'data'.

import pandas as pd
import numpy as np

data = pd.read_csv('/content/drive/MyDrive/ColabNotebooks/bank_marketing.csv')

data.head()


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,31,self-employed,married,tertiary,no,2666,no,no,cellular,10,nov,318,2,97,6,success,yes
1,29,unemployed,single,unknown,no,1584,no,no,cellular,6,sep,245,1,-1,0,unknown,yes
2,41,blue-collar,married,secondary,no,2152,yes,no,cellular,17,nov,369,1,-1,0,unknown,no
3,50,blue-collar,married,secondary,no,84,yes,no,cellular,17,jul,18,8,-1,0,unknown,no
4,40,admin.,married,secondary,no,0,no,no,cellular,28,jul,496,2,182,11,success,yes


In [3]:
#5. Obtener la información de dicha base de datos que incluya el número de registros, el total de vairables, tipo de cada varible, cantidad de datos perdidos de cada variable en caso de que existan.

print('Información del dataset')
data.info()

print('\nCantidad de datos perdidos por variable')
print(data.isnull().sum())

Información del dataset
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9000 entries, 0 to 8999
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        9000 non-null   int64 
 1   job        9000 non-null   object
 2   marital    9000 non-null   object
 3   education  9000 non-null   object
 4   default    9000 non-null   object
 5   balance    9000 non-null   int64 
 6   housing    9000 non-null   object
 7   loan       9000 non-null   object
 8   contact    9000 non-null   object
 9   day        9000 non-null   int64 
 10  month      9000 non-null   object
 11  duration   9000 non-null   int64 
 12  campaign   9000 non-null   int64 
 13  pdays      9000 non-null   int64 
 14  previous   9000 non-null   int64 
 15  poutcome   9000 non-null   object
 16  y          9000 non-null   object
dtypes: int64(7), object(10)
memory usage: 1.2+ MB

Cantidad de datos perdidos por variable
age          0
job          0
marita

In [4]:
#6. Transforma las variables categoricas de manera que puedan ser tratadas numericamente. Justifica si utilizar LabelEncoder o OneHotEncoder.

from sklearn.preprocessing import LabelEncoder, OneHotEncoder

le = LabelEncoder()
ohe = OneHotEncoder(drop = 'first', sparse_output = False)

data_transf = data.copy()

# Usamos LabelEncoder para columnas binarias

col_binarias = ['default', 'housing', 'loan', 'y']

for col in col_binarias:
  data_transf[col] = le.fit_transform(data_transf[col])

# Hacemos transformación manual para education

edu = {'unknown' : 0, 'primary' : 1, 'secondary' : 2, 'tertiary' : 3}

data_transf['education'] = data_transf['education'].map(edu)

# Usamos OneHotEncoder para el resto de las variables categóricas

col_cat = ['job', 'marital', 'contact', 'month', 'poutcome']

datos_ohe = ohe.fit_transform(data_transf[col_cat])

ohe_df = pd.DataFrame(datos_ohe, columns = ohe.get_feature_names_out(col_cat)).astype(int)

data_transf = pd.concat([data_transf.drop(col_cat, axis = 1), ohe_df], axis = 1)

data_transf.head()


,age,education,default,balance,housing,loan,day,duration,campaign,pdays,...,month_jul,month_jun,month_mar,month_may,month_nov,month_oct,month_sep,poutcome_other,poutcome_success,poutcome_unknown
0,31,3,0,2666,0,0,10,318,2,97,...,0,0,0,0,1,0,0,0,1,0
1,29,0,0,1584,0,0,6,245,1,-1,...,0,0,0,0,0,0,1,0,0,1
2,41,2,0,2152,1,0,17,369,1,-1,...,0,0,0,0,1,0,0,0,0,1
3,50,2,0,84,1,0,17,18,8,-1,...,1,0,0,0,0,0,0,0,0,1
4,40,2,0,0,0,0,28,496,2,182,...,1,0,0,0,0,0,0,0,1,0


Se utilizó LabelEncoder en variables con orden jerarquíco o binarias ya que permite representar de forma numérica las relaciones ordinales sin aumentar la dimensionalidad del conjunto de datos. Por otro lado, se aplicó OneHotEncoder en las variables en las que las categorías no presentan una jerarquía natural, esto para evitar introducir relaciones numéricas artificales. En el caso de la variable education se utilizó un mapeo manual para poder asignarle el 0 a unkown.

In [5]:
#7. Transforma las variables numericas en los casos que se tenga algun tipo de sesgo.

col_num = ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']

print('Sesgo de las variables númericas:')
print(data[col_num].skew())

from sklearn.preprocessing import PowerTransformer

pt = PowerTransformer(method = 'yeo-johnson')

data_transf[col_num] = pt.fit_transform(data_transf[col_num])

print('\nSesgo de las variables númericas transformadas:')
print(data_transf[col_num].skew())

Sesgo de las variables númericas:
age         0.801429
balance     7.280036
day         0.117315
duration    2.184045
campaign    5.392712
pdays       2.349177
previous    7.682286
dtype: float64

Sesgo de las variables númericas transformadas:
age         0.012786
balance     1.868257
day        -0.153963
duration    0.002658
campaign    0.263279
pdays       1.182386
previous    1.194494
dtype: float64


In [6]:
#8. Considera la variable 'y' como la variable de salida y el resto de las variables como las variables de entrada.

y = data_transf['y']
X = data_transf.drop('y', axis = 1)

print(y.shape)
print(X.shape)


(9000,)
(9000, 40)


In [7]:
#9. Particiona los datos en los conjuntos de entrenamiento, validación y prueba en 60%, 20% y 20% respectivamente.

from sklearn.model_selection import train_test_split

X_train, X_validation_and_test, y_train, y_validation_and_test = train_test_split(X, y, train_size = 0.6, random_state = 42)
X_validation, X_test, y_validation, y_test = train_test_split(X_validation_and_test, y_validation_and_test, test_size = 0.5, random_state = 42)

print(f"Entrenamiento (60%): {X_train.shape[0]} registros")
print(f"Validación (20%):    {X_validation.shape[0]} registros")
print(f"Prueba (20%):        {X_test.shape[0]} registros")

Entrenamiento (60%): 5400 registros
Validación (20%):    1800 registros
Prueba (20%):        1800 registros


In [8]:
#10. Aplica el modelo de Regresón Logística en el conjunto de entrenamiento. Valida el modelo con las predicciones del conjunto de validación y su matriz de confusión. Ajusta los parámetros del modelo hasta obtener tu mejor resultado.

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import GridSearchCV

#Modelo base

lr = LogisticRegression(C = 1, solver = 'newton-cg', max_iter = 1000)
modelo_lr = lr.fit(X_train, y_train)

print(f"Exactitud del modelo con el conjunto de validación:", modelo_lr.score(X_validation, y_validation))

pr = modelo_lr.predict(X_validation)
print(confusion_matrix(y_validation, pr))

#Busqueda hiperparametros

parametros_grid = [
    {'C' : [0.01, 0.1, 1, 10, 100],
    'solver' : ['liblinear', 'lbfgs', 'newton-cg'],
    'penalty' : ['l2']},
    {'C' : [0.01, 0.1, 1, 10, 100],
    'solver' : ['liblinear'],
    'penalty' : ['l1']},
    {'C' : [0.01, 0.1, 1, 10, 100],
    'solver' : ['saga'],
    'penalty' : ['elasticnet'],
     'l1_ratio' : [0.25, 0.5, 0.75]}
]

grid = GridSearchCV(LogisticRegression(max_iter = 10000), parametros_grid, cv = 5, scoring = 'accuracy', n_jobs = -1)

grid.fit(X_train, y_train)

print('Mejores parámetros:', grid.best_params_)
print('Mejor score:', grid.best_score_)

best_lr = grid.best_estimator_
pred = best_lr.predict(X_validation)

#Modelo con mejores parámetros

print('Matriz de confusión de validación con los mejores parámetros')
print(confusion_matrix(y_validation, pred))
print('Mejor score de validación con mejores parámetros:', best_lr.score(X_validation, y_validation))


Exactitud del modelo con el conjunto de validación: 0.82
[[899 159]
 [165 577]]
Mejores parámetros: {'C': 1, 'penalty': 'l1', 'solver': 'liblinear'}
Mejor score: 0.8227777777777778
Matriz de confusión de validación con los mejores parámetros
[[901 157]
 [168 574]]
Mejor score de validación con mejores parámetros: 0.8194444444444444


In [9]:
#11. Aplica el modelo Red Neuronal en el conjunto de entrenamiento. Valida el modelo con las predicciones del conjunto de validación y su matriz de confusión. Ajusta los parámetros del modelo hasta obtener tu mejor modelo, entre ellos el número de neuronas y capas ocultas.

from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_validation_scaled = scaler.transform(X_validation)

#Modelo Base

mlp = MLPClassifier(hidden_layer_sizes = (15,4), activation = 'relu', solver = 'adam', max_iter = 1000)

mlp.fit(X_train_scaled, y_train)

pred_val = mlp.predict(X_validation_scaled)

print('Exactitud:', mlp.score(X_validation_scaled, y_validation))
print(confusion_matrix(y_validation, pred_val))

#Busqueda hiperparametros

param_grid = {
    'hidden_layer_sizes' : [
        (15,4),
        (50,50),
        (100, ),
    ],
    'activation' : ['tanh', 'relu'],
    'solver' : ['adam'],
    'alpha' : [0.0001, 0.001, 0.01]
}

grid2 = GridSearchCV(MLPClassifier(max_iter = 1000, early_stopping = True), param_grid, cv = 3, scoring = 'accuracy', n_jobs = -1)

grid2.fit(X_train_scaled, y_train)

print('Mejores parámetros:', grid2.best_params_)
print('Mejor score:', grid2.best_score_)

#Validacion del modelo con mejores parámetros

best_mlp = grid2.best_estimator_
pred_val = best_mlp.predict(X_validation_scaled)

print('Mejor score de validación con mejores parámetros mlp:', best_mlp.score(X_validation_scaled, y_validation))
print('Matriz de confusión de validación con los mejores parámetros mlp')
print(confusion_matrix(y_validation, pred_val))


Exactitud: 0.8216666666666667
[[898 160]
 [161 581]]
Mejores parámetros: {'activation': 'relu', 'alpha': 0.01, 'hidden_layer_sizes': (100,), 'solver': 'adam'}
Mejor score: 0.8340740740740741
Mejor score de validación con mejores parámetros mlp: 0.8311111111111111
Matriz de confusión de validación con los mejores parámetros mlp
[[913 145]
 [159 583]]


In [10]:
#12. Selecciona el mejor modelo encontrado en los incisos anteriores y utiliza el conjunto de prueba para obtener el desempeño final del modelo y su matriz de confusión.

pred_test_mlp = best_mlp.predict(scaler.transform(X_test))

print('Mejor score de prueba con mejores parámetros mlp:', best_mlp.score(scaler.transform(X_test), y_test))
print('Matriz de confusión de prueba con los mejores parámetros mlp')
print(confusion_matrix(y_test, pred_test_mlp))

Mejor score de prueba con mejores parámetros mlp: 0.8394444444444444
Matriz de confusión de prueba con los mejores parámetros mlp
[[857 148]
 [141 654]]


La aplicación de tecnicas de inteligencia artificial pueden ser herramientas altamente efectivas para resolver problemas de mercadotecnia, ya que permiten identificar patrones de comportamiento en clientes y predecir con mayor precisión la respuesta de los clientes, en este caso, si adquiriran un plan de inversión ofrecida por el banco. La inteligencia artificial facilita una toma de decisiones más estratégica y basada en datos, mejorando el impacto de los programas de telemarketing.